# Part 2: Write an LRU policy

In Part 1, you saw that agentic LLM calls share long prompt prefixes. An inference engine can retain KV-cache blocks after a call finishes so subsequent calls with matching prefixes can skip redundant prefill computation. However, retained cache blocks consume memory that active calls also need.

In this part, you will implement a least-recently-used (LRU) eviction policy and explore how cache eviction, concurrency caps, batch budgets, and dispatch order affect end-to-end job latency. Policies implement the `Policy` interface in `agentkv/policy.py`, which provides eight hooks (`target_concurrency` and `choose_victim` are required; the others are optional and default to FCFS dispatch/admission, least-loaded routing, and Vidur's vLLM V1 scheduler with a 2048-token mixed batch budget):

```python
class MyPolicy(Policy):
    def on_event(self, event, state): ...                              # observe events
    def target_concurrency(self, view, state) -> int: ...              # set the cap (1–16)
    def choose_victim(self, view, candidates, state) -> str: ...       # evict a cached block
    def dispatch_order(self, view, ready, state) -> list[str]: ...     # order ready calls
    def admission_order(self, view, waiting, state) -> list[str]: ...  # order engine admissions
    def route_call(self, view, call, state) -> int: ...                # select a replica
    def batch_token_budget(self, view, state) -> int: ...              # choose the next batch's budget
    def choose_preempted(self, view, running, state) -> str: ...       # preempt a running call
```

Complete the `[Part 2] TODO` sections in `policies/reference.py` to initialize your recency state, update it in `on_event`, and evict the least recently used candidate in `choose_victim`. See the [Part 2 handout](../docs/part2_handout.md) and `agentkv/policy.py` for full hook specifications.

![Policy hooks in the application and inference engine.](../docs/figures/system.svg)

**Estimated time:** about 1 hour.

**Grading (5 points):** LRU correctness on hidden test cases (use the public oracle and invariant tests in section 2 during development). Sections 3–7 are ungraded exploration to prepare you for Part 3.

## Evaluator setup

Part 2 simulates a single model replica (Llama-3-8B on one NVIDIA A100 80 GB GPU) with 16-token KV-cache blocks and Vidur execution-time predictions. Each replay fixes the workload, KV-cache quota, and batch limits, starting from an empty cache. (Part 3 uses two replicas; always compare policies under the same replica count. See [Evaluator configuration](../README.md#evaluator-configuration) for details.)

## Before you begin

Complete the [environment setup](../README.md#getting-started) before running this notebook, and select **CS 378 Assignment 4** as the notebook kernel.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from agentkv import viz
from agentkv.notebook import check, repo_root

os.chdir(repo_root())          # every path below is relative to the repository root
plt.rcParams["figure.dpi"] = 100

from agentkv import score
from agentkv.harness import BlockHarness
from agentkv.policy import Policy
from agentkv.sandbox.loader import instantiate, load_policy_class


def load_policy(path, **params):
    """Instantiate the policy class defined in a policy file."""
    return instantiate(load_policy_class(Path(path).read_text(), path), params)


def show(h, note=""):
    """Print the cache state: [A*] is active, [B] is cached, and [ ] is empty."""
    active, cached = h.active(), h.reusable()
    slots = [f"[{n}*]" for n in active] + [f"[{n}]" for n in cached] + ["[ ]"] * h.empty_slots()
    evicted = f"   evicted so far: {', '.join(h.evicted)}" if h.evicted else ""
    print(f"{' '.join(slots):24} {note}{evicted}")

## 1. KV-cache allocation and reuse

Within a replica, prompts that share a common prefix can reuse the full 16-token KV blocks for that prefix:

![Three prompts and the prefix blocks they share.](../docs/figures/kv_prefix.svg)

To trace block allocation and eviction step by step, we use `BlockHarness`, a small teaching cache with letter-labeled 1-token blocks:

- **Active (`[A*]`):** Held by at least one running call (`refs > 0`). Active blocks can never be evicted.
- **Cached (`[A]`):** Released by all calls (`refs == 0`), but retained in memory so a future call with the same prefix can reuse it (`hit`). Only cached blocks appear in `candidates` when `choose_victim` is called.
- **Empty (`[ ]`):** Unused slot. The engine always fills empty slots before calling `choose_victim`.

The cell below runs a 3-slot cache under FIFO eviction (`candidates` are passed oldest-released first, so `candidates[0]` is the block whose final release happened earliest). Step through the output to see why allocating `D` triggers an eviction while earlier allocations do not.

In [ ]:
class FIFO(Policy):
    def target_concurrency(self, view, state):
        return 4

    def choose_victim(self, view, candidates, state):
        return candidates[0]          # candidates arrive oldest-released first


h = BlockHarness(FIFO(), capacity=3)
h.allocate("A")
h.allocate("B")
show(h, "allocate A, B")

h.release("A")
h.release("B")
show(h, "release both")

h.allocate("C")
h.release("C")
show(h, "C takes the empty slot")

h.access("A")
h.release("A")
show(h, "hit on A, then A is released again")

h.allocate("D")
show(h, "D needs a slot: FIFO evicts the oldest release, B")

In the trace above, FIFO and LRU evict the same block (`B`) because `A` was both accessed and released most recently.

However, **release order (FIFO)** and **access order (LRU)** differ when a block is accessed more recently, but finishes and releases its reference *earlier* than an older block:

1. Call 1 allocates `A` (access order: `A`).
2. Call 2 allocates `B`, and Call 3 also hits `B` (access order: `A`, then `B`).
3. Calls 2 and 3 finish first, releasing `B`, while Call 1 finishes later, releasing `A` (release order: `B`, then `A`).

When the cache later needs space for `D`, which block will FIFO evict, and which block would LRU evict? Run the cell below to see FIFO's behavior.

In [ ]:
h = BlockHarness(FIFO(), capacity=3)
h.allocate("A")
h.allocate("B")
h.access("B")
show(h, "A held by one call, B by two")

h.release("B")
h.release("B")
h.release("A")
show(h, "B's calls finish first, then A's")

h.allocate("C")
h.allocate("D")
show(h, "FIFO evicts B, even though B was accessed more recently than A")

## 2. Implement LRU eviction

Now implement LRU eviction in `policies/reference.py` by completing `__init__`, `on_event`, and `choose_victim`:

- **`__init__`:** Initialize a data structure (such as ` collections.OrderedDict`) to track block IDs from least recently used to most recently used.
- **`on_event(self, event, state)`:**
  - When `event.kind in ("alloc", "hit")`, record `event.block` as the most recently used block (e.g., `move_to_end(event.block)`). Always follow event delivery order rather than comparing timestamps, since multiple events can share a timestamp.
  - When `event.kind == "evict"`, remove `event.block` from your recency tracker.
  - Ignore `"release"` and all other event kinds—releasing a block does not count as an access.
- **`choose_victim(self, view, candidates, state)`:**
  - Return the block ID in `candidates` that was least recently used according to your tracker. (Note: your tracker may also contain currently active blocks, which will not be in `candidates`.)

`policies/reference.py` also serves as the baseline starting policy for Part 3 (with default `cap = 8`). Run the unit tests and the 3-slot trace below to verify that your LRU policy evicts `A` instead of `B`.

In [ ]:
check("tests/test_part2_lru.py", "tests/test_kv_invariants.py")

In [ ]:
h = BlockHarness(load_policy("policies/reference.py"), capacity=3)
h.allocate("A")
h.allocate("B")
h.access("B")

h.release("B")
h.release("B")
h.release("A")

h.allocate("C")
h.allocate("D")
show(h, "LRU evicts A: B was used more recently")

## 3. Evaluate LRU on agent traces

With LRU working, we can replay full workload traces in the Vidur simulator using `viz.record` and inspect how concurrency caps interact with KV-cache capacity.

We start with `dev-W3-0`, a **document QA** trace where users ask short questions about ~12,000-token documents from a shared library and the model generates short answers. In the two plots below (`cap = 8`):

- **Top plot (`viz.timeline`):** One row per job. **Yellow** = ready call waiting for a concurrency slot (`target_concurrency`); **Orange** = dispatched call waiting in the replica queue for engine admission; **Blue** = admitted call running prefill/decode; **Black tick (`|`)** = job arrival.
- **Bottom plot (`viz.memory`):** Stacked KV-cache block usage over time (**Blue** = active blocks held by running calls; **Green** = cached blocks available for prefix reuse; **Gray** = empty blocks).

In [ ]:
W3 = "data/traces/public_dev/dev-W3-0.json"
rec8 = viz.record(W3, "policies/reference.py", params={"cap": 8})

fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [2.4, 1]})
viz.timeline(rec8, axes[0])
viz.memory(rec8, axes[1])
fig.tight_layout()
plt.show()

Look at the memory plot above: when 8 long-context calls run concurrently, active blocks (blue) consume almost the entire KV cache, leaving very little room for cached blocks (green) and forcing frequent evictions.

Next, replay `dev-W3-0` across `cap` values of `8, 4, 3, 2, 1`. Before running the cell, predict how lowering `cap` will trade batching concurrency against cache hit rate and prefill time.

In [ ]:
for cap in (8, 4, 3, 2, 1):
    r = viz.record(W3, "policies/reference.py", params={"cap": cap})
    mean_latency = sum(r.result.flows) / len(r.result.flows)
    hit = sum(c["cached"] for c in r.calls) / sum(c["prompt"] for c in r.calls)
    d = r.result.diag
    print(f"cap {cap}: mean job latency {mean_latency:5.2f} s   prompt tokens from cache {hit:.0%}   "
          f"engine busy {d['busy_time']:4.1f} s, prefilling {d['prefill_time']:4.1f} s")

Now run the same concurrency sweep on `dev-D-0`, a **deep research** trace with the same parallel-branch DAG structure as Part 1. Unlike single-call document QA jobs, each research report fans out into parallel subtopic branches and generates longer extraction and synthesis outputs.

In [ ]:
D = "data/traces/public_dev/dev-D-0.json"
for cap in (8, 4, 2):
    r = viz.record(D, "policies/reference.py", params={"cap": cap})
    mean_latency = sum(r.result.flows) / len(r.result.flows)
    hit = sum(c["cached"] for c in r.calls) / sum(c["prompt"] for c in r.calls)
    d = r.result.diag
    print(f"cap {cap}: mean job latency {mean_latency:5.2f} s   prompt tokens from cache {hit:.0%}   "
          f"engine busy {d['busy_time']:4.1f} s, prefilling {d['prefill_time']:4.1f} s")

Compare the two workloads:

- **Document QA (`dev-W3-0`):** Every call has a ~12K-token prompt and a tiny output. Lowering `cap` reserves more cache space for shared documents, raising the cache hit rate and cutting redundant prefill work.
- **Deep research (`dev-D-0`):** Each report has parallel branches on its critical path and longer decode phases. Restricting `cap` too aggressively serializes parallel branches and increases slot-wait time. (Note: because Vidur batches prefill and decode together, `prefill_time` counts any batch step that includes prefill tokens.)

## 4. Compare across all eight workload families

Now evaluate your LRU policy across all 32 development traces (`public_dev`, 4 traces per family) on one replica. For each family, `score()` computes the geometric mean speedup (`reference mean job latency / candidate mean job latency`), and then takes the geometric mean across all eight families (`S = 1.000` at the reference default `cap = 8`; `S > 1` is faster).

![The eight workload families used for evaluation.](../docs/figures/families.svg)

Run the sweep below to see how each workload family responds to concurrency caps from `1` to `16`.

In [ ]:
CAPS = (1, 2, 3, 4, 6, 8, 12, 16)
sweep = {cap: score("policies/reference.py", split="public_dev", replicas=1, params={"cap": cap})
         for cap in CAPS}
if not sweep[8].valid:
    raise ValueError(f"policies/reference.py is invalid: {sweep[8].violations[0]}")
families = list(sweep[8].by_family)
grid = [[sweep[c].by_family[f] for c in CAPS] for f in families]

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(grid, cmap="RdBu", vmin=0.5, vmax=1.5, aspect="auto")
ax.set_xticks(range(len(CAPS)), CAPS)
ax.set_yticks(range(len(families)), families)
ax.set_xlabel("cap")
ax.set_title("LRU speedup relative to the baseline (blue: faster)")
for i, row in enumerate(grid):
    for j, v in enumerate(row):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im)
plt.show()
print("S by cap:", {c: round(r.speedup, 3) for c, r in sweep.items()})

## 5. Eviction order and prefix tree structure

First, let's compare `policies/fifo.py` and `policies/reference.py` across `public_dev` at the same concurrency cap (`cap = 8`).

In [ ]:
for path in ("policies/fifo.py", "policies/reference.py"):
    result = score(path, split="public_dev", replicas=1)
    print(Path(path).name, result.valid, f"S={result.speedup:.3f}", result.by_family)

Why does FIFO sometimes match or even beat basic LRU on certain families? The answer lies in how **prefix trees** are accessed and matched in the KV cache.

![Evicting a root prevents reuse of its descendants; evicting a tail preserves the remaining prefix.](../docs/figures/evict_tail.svg)

Suppose a prompt consists of blocks `B0 -> B1 -> B2 -> B3` (from root to tail):

1. **Access order (root to tail):** When a call reuses or allocates this prompt, the engine accesses `B0` first, then `B1`, `B2`, and finally `B3`. Basic LRU therefore marks the **root (`B0`) as the oldest (least recently used)** block among the four, and the **tail (`B3`) as the newest**.
2. **Prefix matching stops at the first gap:** On a future call, the engine matches prefix blocks starting from `B0`. If basic LRU evicts `B0`, then `B1`, `B2`, and `B3` become **unreachable orphans**—even though they are still sitting in cache memory, no future prompt can match them without `B0`!
3. **Why FIFO avoids this:** When a call finishes, Vidur releases its blocks from **tail to root** (`B3`, then `B2`, `B1`, `B0`). FIFO therefore naturally evicts tail blocks (`B3`) first, preserving the usable prefix `B0 -> B1 -> B2`.

### Try tail-aware LRU (`candidates/lru_tails.py`)

Each resident block in `view.blocks` exposes:
- `block.id`: physical slot ID (what `candidates` contains and `choose_victim` returns)
- `block.prefix`: content hash of the prefix up to this block
- `block.parent`: content hash of the preceding block (`None` for a root block)
- `block.replica`: which replica holds this block

A cached block is a **leaf (tail)** on its replica if no other resident block on the same replica has `other.parent == block.prefix`. Copy `policies/reference.py` to `candidates/lru_tails.py` and modify `choose_victim` so that it prefers evicting the least recently used *leaf* candidate (falling back to regular LRU if needed). Keep `policies/reference.py` unchanged for Part 2 grading. (You can also experiment with overriding `batch_token_budget` to return `512`, `1024`, `2048`, or `4096`.)

In [ ]:
if not Path("candidates/lru_tails.py").exists():
    print("Optional: copy policies/reference.py to candidates/lru_tails.py and prefer leaf victims.")
else:
    for path in ("policies/reference.py", "candidates/lru_tails.py"):
        r = score(path, split="public_dev", replicas=1, params={"cap": 8})
        print(f"{path:26} S = {r.speedup:.3f}  {r.by_family}")

## 6. Call-level vs. job-level dispatch order

While eviction controls which prefixes stay in memory, `dispatch_order(self, view, ready, state)` controls which ready calls enter the engine next.

To see why dispatch order affects **mean job latency**, consider two 3-call jobs (`Job A` and `Job B`) that both arrive at `t = 0`, where each call takes `1 s` and `cap = 1`:

- **Interleaved dispatch (`A1, B1, A2, B2, A3, B3`):** `Job A` finishes at `t = 5` and `Job B` finishes at `t = 6` $\rightarrow$ **mean job latency = 5.5 s**.
- **Oldest-job-first dispatch (`A1, A2, A3, B1, B2, B3`):** `Job A` finishes at `t = 3` and `Job B` finishes at `t = 6` $\rightarrow$ **mean job latency = 4.5 s** (with the exact same total GPU work).

By default, `ready` calls are dispatched in the order individual calls become ready, which often interleaves calls across many active jobs. Copy your policy to `candidates/lru_jobs.py` and add oldest-job-first dispatch (inspired by Parrot):

```python
    def dispatch_order(self, view, ready, state):
        return sorted(ready, key=lambda c: view.jobs[view.ready[c].job].arrival)
```

Because Python's `sorted` is stable, calls belonging to the same job remain in readiness order. Run the cell below to compare per-family speedups.

In [ ]:
base = "candidates/lru_tails.py" if Path("candidates/lru_tails.py").exists() else "policies/reference.py"
if not Path("candidates/lru_jobs.py").exists():
    print("Optional: copy your policy to candidates/lru_jobs.py and add dispatch_order.")
else:
    before = score(base, split="public_dev", replicas=1, params={"cap": 8})
    after = score("candidates/lru_jobs.py", split="public_dev", replicas=1, params={"cap": 8})
    print(f"{'family':8} {base:>24} {'candidates/lru_jobs.py':>24}")
    for f in before.by_family:
        print(f"{f:8} {before.by_family[f]:24.3f} {after.by_family[f]:24.3f}")
    print(f"{'S':8} {before.speedup:24.3f} {after.speedup:24.3f}")

## 7. Prepare for Part 3 policy research

In Part 3, `score()` evaluates policies across **two independent replicas** (`replicas=2`). Each replica has its own waiting queue and KV cache, and `route_call(self, view, call, state)` selects which replica receives each dispatched call (defaulting to the replica with the fewest unfinished calls). Using `view.replicas` and `view.blocks`, your policies can trade prefix affinity (routing a call to the replica that already caches its prefix) against load balancing.

Make sure `policies/reference.py` passes `pytest tests/test_part2_lru.py tests/test_kv_invariants.py`, and include it in your team's Canvas archive.